# Modeling

This notebook trains and evaluates models that predict point outcome using only pre-serve information.

The goal is not just accuracy, but interpretability: understanding which serve features and match contexts are most associated with winning points.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

df = pd.read_csv("../data/processed/table_tennis_serves_features.csv")
df.head()

In [ ]:
model_features = ["serve_type","spin_type","spin_intensity","serve_length","placement_zone","toss_height","contact_point","game_number","server_score","receiver_score","game_state","opponent_skill_level","opponent_style","side","intended_setup","score_margin","total_points_played_in_game","is_tied","is_trailing","is_leading","is_late_game","is_deuce_or_later","is_game_point_for_server","is_game_point_against_server","serve_spin_combo","serve_length_spin_combo","serve_placement_combo","full_serve_combo","is_heavy_spin","is_low_spin","combo_attempts","combo_win_rate","combo_reliability"]
X = df[model_features]
y = df["point_won"]
groups = df["match_id"]

The model excludes post-serve variables such as return quality, rally length, and point-end type to avoid data leakage.

In [ ]:
baseline_accuracy = y.value_counts(normalize=True).max()
baseline_accuracy

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),("num", StandardScaler(), numeric_features)])

In [ ]:
cv = GroupKFold(n_splits=5)

GroupKFold is used so that points from the same match do not appear in both the training and validation sets. This creates a more realistic estimate of model performance.

In [ ]:
lasso_model = Pipeline(steps=[("preprocessor", preprocessor),("classifier", LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, class_weight="balanced"))])
lasso_accuracy = cross_val_score(lasso_model, X, y, cv=cv, groups=groups, scoring="accuracy")
lasso_auc = cross_val_score(lasso_model, X, y, cv=cv, groups=groups, scoring="roc_auc")
print("LASSO Accuracy:", lasso_accuracy.mean())
print("LASSO ROC-AUC:", lasso_auc.mean())

In [ ]:
lasso_model.fit(X, y)
feature_names = lasso_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = lasso_model.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
important_features = coef_df[coef_df["coefficient"] != 0].sort_values("abs_coefficient", ascending=False)
important_features.head(20)

Positive coefficients are associated with higher predicted probability of winning the point. Negative coefficients are associated with lower predicted probability.

In [ ]:
rf_model = Pipeline(steps=[("preprocessor", preprocessor),("classifier", RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42, class_weight="balanced"))])
rf_accuracy = cross_val_score(rf_model, X, y, cv=cv, groups=groups, scoring="accuracy")
rf_auc = cross_val_score(rf_model, X, y, cv=cv, groups=groups, scoring="roc_auc")
print("Random Forest Accuracy:", rf_accuracy.mean())
print("Random Forest ROC-AUC:", rf_auc.mean())

In [ ]:
model_results = pd.DataFrame({"model":["Baseline","LASSO Logistic Regression","Random Forest"],"accuracy":[baseline_accuracy,lasso_accuracy.mean(),rf_accuracy.mean()],"roc_auc":[np.nan,lasso_auc.mean(),rf_auc.mean()]})
model_results

The final recommendation system uses the model with the strongest balance of predictive performance and interpretability.

If the random forest performs better, it is used for predicted win probability. The LASSO model remains useful for interpretation because it shows which serve and context variables are most strongly associated with point outcomes.

In [ ]:
import joblib
os.makedirs("../models", exist_ok=True)
rf_model.fit(X, y)
joblib.dump(rf_model, "../models/serve_win_probability_model.pkl")
joblib.dump(model_features, "../models/model_features.pkl")